# analyze_page — research notebook

The first **LLM node**. Takes `clean_text`, returns the concepts worth testing plus a sufficiency judgment. One structured-output call — the model fills a schema, it calls no tools.

```
analyze_page(clean_text, title, question_count) -> AnalyzePageResult(ok, concepts, sufficiency_score, max_supportable_questions, error)
```

**Resilience rule: never raises.** LLM/auth/network failures return `ok=False`.

Needs `OPENAI_API_KEY` in `research/.env`.

In [1]:
# pip install -q langchain-openai python-dotenv   (in research/requirements.txt)
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(".env")

class Concept(TypedDict):
    name: str
    salience: float        # 0..1, how central to the article
    supporting_span: str   # short verbatim quote that grounds the concept

class PageAnalysis(TypedDict):
    concepts: list[Concept]
    sufficiency_score: float          # 0..1: can this text support the request?
    max_supportable_questions: int

class AnalyzePageResult(TypedDict):
    ok: bool
    concepts: list[Concept]
    sufficiency_score: float
    max_supportable_questions: int
    error: str

llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-5-mini"))
analyzer = llm.with_structured_output(PageAnalysis)

## Prompt + node

The `supporting_span` quotes matter: `critique_questions` later verifies grounding against them, so we also verify each span really appears in the text and drop any the model hallucinated.

In [2]:
SYSTEM = """You are analyzing an article to prepare a quiz.
Extract the concepts worth testing: key ideas, claims, mechanisms and relationships. Not trivia.
For each concept, copy a short VERBATIM quote from the article as supporting_span.
Rate each concept's salience (0..1, how central it is to the article).
Then judge sufficiency for {n} questions. Note: one rich concept can support several
distinct questions at different cognitive levels (define it, apply it, analyze it),
so max_supportable_questions counts question opportunities, not concepts. It is limited
only by how much the text actually says: never count questions that would need facts
beyond the article."""

import re as _re

def _norm(s: str) -> str:
    return _re.sub(r"\s+", " ", s).strip().lower()

def analyze_page(clean_text: str, title: str, question_count: int) -> AnalyzePageResult:
    """Never raises: LLM failures return ok=False."""
    fail = dict(concepts=[], sufficiency_score=0.0, max_supportable_questions=0)
    try:
        out: PageAnalysis = analyzer.invoke([
            ("system", SYSTEM.format(n=question_count)),
            ("user", f"Title: {title}\n\nArticle:\n{clean_text}"),
        ])
        # trust but verify (whitespace/case tolerant): drop hallucinated spans, clamp salience
        text_n = _norm(clean_text)
        grounded: list[Concept] = []
        for con in out["concepts"]:
            if _norm(con["supporting_span"]) in text_n:
                con["salience"] = min(1.0, max(0.0, float(con["salience"])))
                grounded.append(con)
        if not grounded:
            return AnalyzePageResult(ok=False, error="Model returned no grounded concepts.", **fail)
        return AnalyzePageResult(ok=True, concepts=grounded,
                                 sufficiency_score=min(1.0, max(0.0, float(out["sufficiency_score"]))),
                                 max_supportable_questions=max(0, int(out["max_supportable_questions"])),
                                 error="")
    except Exception as e:                       # noqa: BLE001 — resilience boundary
        return AnalyzePageResult(ok=False, error=f"Analysis failed: {e}", **fail)

## Try it

A short sample first (cheap); swap in `clean_page` output from any URL once your key is set.

In [3]:
SAMPLE = (
    "Photosynthesis converts light energy into chemical energy. In the light-dependent "
    "reactions, thylakoid membranes produce ATP and NADPH. The Calvin cycle then uses "
    "these to fix carbon dioxide into glucose in the stroma. Chlorophyll a absorbs most "
    "strongly at 430 nm and 662 nm, reflecting green light, which is why leaves look green. "
    "The oxygen released transformed Earth's atmosphere during the Great Oxidation Event."
)

r = analyze_page(SAMPLE, "Photosynthesis", question_count=5)
if r["ok"]:
    for c in sorted(r["concepts"], key=lambda c: -c["salience"]):
        print(f'{c["salience"]:.2f}  {c["name"]}  |  "{c["supporting_span"][:60]}"')
    print("sufficiency:", r["sufficiency_score"], "| max questions:", r["max_supportable_questions"])
else:
    print("graceful failure:", r["error"])

1.00  Definition of photosynthesis  |  "Photosynthesis converts light energy into chemical energy."
0.95  Light-dependent reactions produce ATP and NADPH  |  "In the light-dependent reactions, thylakoid membranes produc"
0.95  Calvin cycle fixes CO2 into glucose using ATP and NADPH  |  "The Calvin cycle then uses these to fix carbon dioxide into "
0.85  Chlorophyll a absorption spectrum and leaf color  |  "Chlorophyll a absorbs most strongly at 430 nm and 662 nm, re"
0.70  Location of light-dependent reactions (thylakoid membranes)  |  "thylakoid membranes"
0.70  Location of the Calvin cycle (stroma)  |  "in the stroma"
0.65  Oxygen release altered Earth's atmosphere (Great Oxidation Event)  |  "The oxygen released transformed Earth's atmosphere during th"
sufficiency: 1.0 | max questions: 9


In [4]:
# resilience: nonsense input still returns a structured result, never an exception
weak = analyze_page("Buy now! 50% off!", "Ad", question_count=10)
print(weak["ok"], "|", weak.get("error") or f"sufficiency={weak['sufficiency_score']}")

True | sufficiency=1.0


## Notes

- **Node, not agent step:** the LLM fills a schema; the graph routes on the result — `ok=False` → end_gracefully, low `sufficiency_score` → adjust_scope, else → plan_quiz.
- **max_supportable_questions counts question opportunities, not concepts** — one rich concept can be asked at several Bloom levels (the question-count rule: min(requested, cap 20, supportable), variants before shrinking, never pad).
- **Span verification is deterministic** and whitespace/case tolerant; hallucinated quotes are dropped before they can poison critique_questions. Salience and scores are clamped to their ranges.
- `with_structured_output` accepts the TypedDict directly; the same types move into the app's state module.
- Cost control: this call sees up to 6k words once per run; keep the model small (gpt-5-mini) — generation needs the stronger model, analysis does not.
- Tune later: does salience help plan_quiz more than concept order alone? Measure once real usage exists.